# Test Metrics Summary

This notebook reads a `test_metrics_over_all.csv` file and prints mean/std per metric, overall and by body part.
Use the utility functions below to load a CSV and display results for multiple models in the same notebook.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

def load_metrics_csv(csv_path: Path) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    # The file has a header row: file_name, MAE, MSE, PSNR, SSIM[, mask_voxels]
    # Some files may include a leading unnamed index column; drop it if present.
    df = pd.read_csv(csv_path)

    if "file_name" not in df.columns:
        df = df.iloc[:, 1:]

    if "file_name" not in df.columns:
        raise ValueError("Could not find 'file_name' column after dropping index column.")

    if df.shape[1] == 5:
        df.columns = ["file_name", "MAE", "MSE", "PSNR", "SSIM"]
    elif df.shape[1] == 6:
        df.columns = ["file_name", "MAE", "MSE", "PSNR", "SSIM", "mask_voxels"]

    # Convert metrics to numeric (invalid values become NaN)
    metrics = ["MAE", "MSE", "PSNR", "SSIM"]
    for col in metrics:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if "mask_voxels" in df.columns:
        df["mask_voxels"] = pd.to_numeric(df["mask_voxels"], errors="coerce")

    return df

def load_volume_metrics_csv(volume_csv_path: Path) -> pd.DataFrame:
    if not volume_csv_path.exists():
        raise FileNotFoundError(f"CSV not found: {volume_csv_path}")

    df = pd.read_csv(volume_csv_path)

    if "volume_id" not in df.columns:
        df = df.iloc[:, 1:]

    if "volume_id" not in df.columns:
        raise ValueError("Could not find 'volume_id' column after dropping index column.")

    metrics = ["MAE", "MSE", "PSNR", "SSIM"]
    for col in metrics:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "mask_voxels" in df.columns:
        df["mask_voxels"] = pd.to_numeric(df["mask_voxels"], errors="coerce")
    if "slice_count" in df.columns:
        df["slice_count"] = pd.to_numeric(df["slice_count"], errors="coerce")

    return df

def format_mean_std(summary: pd.DataFrame, mean_col: str, std_col: str, out_name: str) -> pd.DataFrame:
    formatted = summary.apply(
        lambda row: f"{row[mean_col]:.6g} ± {row[std_col]:.6g}",
        axis=1,
    ).to_frame(name=out_name)
    return formatted

def overall_mean_std(df: pd.DataFrame) -> pd.DataFrame:
    metrics = ["MAE", "MSE", "PSNR", "SSIM"]
    overall_mean = df[metrics].mean()
    overall_std = df[metrics].std()
    overall_summary = pd.DataFrame({"mean": overall_mean, "std": overall_std})
    return format_mean_std(overall_summary, "mean", "std", "mean ± std")

def add_body_part(df: pd.DataFrame, col: str = "file_name") -> pd.DataFrame:
    # Body part grouping derived from filename prefix (e.g., AB_*, HN_*)
    # Example filename: AB_1ABC127-99.nii -> body_part = AB
    df = df.copy()
    df["body_part"] = df[col].astype(str).str.split("_").str[0]
    return df

def mean_std_by_body_part(df: pd.DataFrame) -> pd.DataFrame:
    metrics = ["MAE", "MSE", "PSNR", "SSIM"]
    grouped = df.groupby("body_part")[metrics].agg(["mean", "std"])

    # Reformat to mean ± std per metric
    formatted = pd.DataFrame(index=grouped.index)
    for metric in metrics:
        mean_col = (metric, "mean")
        std_col = (metric, "std")
        formatted[metric] = grouped.apply(
            lambda row: f"{row[mean_col]:.6g} ± {row[std_col]:.6g}",
            axis=1,
        )
    return formatted

def show_metrics_for_csv(csv_path: Path) -> None:
    df = load_metrics_csv(csv_path)

    print("Overall metrics (slice-based):")
    display(overall_mean_std(df))

    df_bp = add_body_part(df, col="file_name")
    print("Metrics by body part (slice-based):")
    display(mean_std_by_body_part(df_bp))

    volume_csv_path = csv_path.parent / "test_metrics_over_volume.csv"
    if volume_csv_path.exists():
        vol_df = load_volume_metrics_csv(volume_csv_path)
        print("Overall metrics (volume-based):")
        display(overall_mean_std(vol_df))

        vol_df_bp = add_body_part(vol_df, col="volume_id")
        print("Metrics by body part (volume-based):")
        display(mean_std_by_body_part(vol_df_bp))
    else:
        print(f"Volume metrics CSV not found: {volume_csv_path}")
        print("Run: python training/scripts/compute_volume_metrics.py <path/to/test_metrics_over_all.csv>")


In [2]:
# Per-slice metrics and influence for a single volume
def show_volume_slice_influence(df: pd.DataFrame, volume=None) -> None:
    # volume can be a volume_id string (e.g., 'AB_1ABA009') or an index into the unique volumes list
    if "file_name" not in df.columns:
        raise ValueError("df must contain a 'file_name' column.")

    def _volume_key(name: str) -> str:
        base = name
        if base.endswith(".nii.gz"):
            base = base[:-7]
        elif base.endswith(".nii"):
            base = base[:-4]
        # Use the part before the last '-' as volume id
        return base.rsplit("-", 1)[0]

    def _slice_key(name: str) -> str:
        base = name
        if base.endswith(".nii.gz"):
            base = base[:-7]
        elif base.endswith(".nii"):
            base = base[:-4]
        parts = base.rsplit("-", 1)
        return parts[1] if len(parts) == 2 else ""

    df = df.copy()
    df["volume_id"] = df["file_name"].astype(str).map(_volume_key)
    df["slice_id"] = df["file_name"].astype(str).map(_slice_key)

    volumes = sorted(df["volume_id"].dropna().unique())
    if not volumes:
        print("No volumes found in df.")
        return

    if volume is None:
        volume_id = volumes[0]
    elif isinstance(volume, int):
        if volume < 0 or volume >= len(volumes):
            raise IndexError(f"volume index out of range: {volume} (0..{len(volumes)-1})")
        volume_id = volumes[volume]
    else:
        volume_id = str(volume)
        if volume_id not in volumes:
            raise ValueError(f"volume_id not found: {volume_id}")

    vol_df = df[df["volume_id"] == volume_id].copy()
    if vol_df.empty:
        print(f"No slices found for volume {volume_id}.")
        return

    metrics_cols = ["MAE", "MSE", "PSNR", "SSIM"]
    vol_df[metrics_cols] = vol_df[metrics_cols].apply(pd.to_numeric, errors="coerce")

    if "mask_voxels" not in vol_df.columns:
        print("No mask_voxels column found; cannot compute volume influence.")
        display(vol_df[["file_name", "slice_id", *metrics_cols]])
        return

    vol_df["mask_voxels"] = pd.to_numeric(vol_df["mask_voxels"], errors="coerce").fillna(0)
    total_mask = vol_df["mask_voxels"].sum()
    if total_mask == 0:
        print("mask_voxels sum to 0 for this volume; cannot compute influence.")
        display(vol_df[["file_name", "slice_id", *metrics_cols, "mask_voxels"]])
        return

    vol_df["weight_pct"] = (vol_df["mask_voxels"] / total_mask) * 100

    # Volume-weighted metric for this volume
    weights = vol_df["mask_voxels"] / total_mask
    weighted_mean = vol_df[metrics_cols].mul(weights, axis=0).sum()

    print(f"Volume: {volume_id} (slices={len(vol_df)}, total_mask={int(total_mask)})")
    display(pd.DataFrame({"weighted_mean": weighted_mean}))

    # Per-slice influence table
    show_cols = ["slice_id", *metrics_cols, "mask_voxels", "weight_pct", "file_name"]
    display(vol_df.sort_values("slice_id")[show_cols])

# Example usage (pick a volume index or id)
# show_volume_slice_influence(df, volume=0)
# show_volume_slice_influence(df, volume="AB_1ABA009")


# CUT

## Abdomen

In [3]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_abdomen_final/test_50/test_metrics_over_all.csv")
df = df = load_metrics_csv(csv_path)


show_volume_slice_influence(df, volume=3)

Volume: AB_1ABA060 (slices=92, total_mask=1532110)


,weighted_mean
MAE,104.049380
MSE,5329.050556
PSNR,35.179822
SSIM,0.696269


,slice_id,MAE,MSE,PSNR,SSIM,mask_voxels,weight_pct,file_name
283,1,149.142246,2873.365371,37.661170,0.458209,13786,0.899805,AB_1ABA060-1.nii
284,10,120.996230,6195.353211,34.324417,0.593022,14589,0.952216,AB_1ABA060-10.nii
285,11,119.479368,6121.445015,34.376539,0.589737,14613,0.953783,AB_1ABA060-11.nii
286,12,118.875727,5989.493050,34.471177,0.555752,14605,0.953261,AB_1ABA060-12.nii
287,13,121.982865,5969.256212,34.485876,0.534117,14648,0.956067,AB_1ABA060-13.nii
...,...,...,...,...,...,...,...,...
370,89,87.826496,3474.748912,36.835844,0.898760,18380,1.199653,AB_1ABA060-89.nii
371,9,117.973014,6296.803938,34.253876,0.597186,14526,0.948104,AB_1ABA060-9.nii
372,90,89.364154,3387.776044,36.945931,0.893433,18267,1.192277,AB_1ABA060-90.nii
373,91,94.957211,3357.120718,36.985409,0.873931,18042,1.177592,AB_1ABA060-91.nii


In [4]:
# Run metrics for the selected CSV path

csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results/cut_synthrad_abdomen_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)


Overall metrics (slice-based):


,mean ± std
MAE,118.834 ± 58.5118
MSE,4360.42 ± 1446.23
PSNR,36.402 ± 3.16541
SSIM,0.602407 ± 0.193639


Metrics by body part (slice-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
AB,118.834 ± 58.5118,4360.42 ± 1446.23,36.402 ± 3.16541,0.602407 ± 0.193639


Overall metrics (volume-based):


,mean ± std
MAE,119.202 ± 29.6626
MSE,4518.64 ± 844.641
PSNR,36.0676 ± 0.691409
SSIM,0.601683 ± 0.090678


Metrics by body part (volume-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
AB,119.202 ± 29.6626,4518.64 ± 844.641,36.0676 ± 0.691409,0.601683 ± 0.090678


### New Script

In [6]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results_new_script/cut_synthrad_abdomen_final/test_50/test_metrics_over_all.csv")
df = df = load_metrics_csv(csv_path)


show_volume_slice_influence(df, volume=3)

Volume: AB_1ABA060 (slices=92, total_mask=1532110)


,weighted_mean
MAE,104.049380
MSE,5329.050556
PSNR,35.179822
SSIM,0.595852


,slice_id,MAE,MSE,PSNR,SSIM,mask_voxels,weight_pct,file_name
283,1,149.142246,2873.365371,37.661170,0.544122,13786,0.899805,AB_1ABA060-1.nii
284,10,120.996230,6195.353211,34.324417,0.541608,14589,0.952216,AB_1ABA060-10.nii
285,11,119.479368,6121.445015,34.376539,0.544955,14613,0.953783,AB_1ABA060-11.nii
286,12,118.875727,5989.493050,34.471177,0.544923,14605,0.953261,AB_1ABA060-12.nii
287,13,121.982865,5969.256212,34.485876,0.535146,14648,0.956067,AB_1ABA060-13.nii
...,...,...,...,...,...,...,...,...
370,89,87.826496,3474.748912,36.835844,0.710596,18380,1.199653,AB_1ABA060-89.nii
371,9,117.973014,6296.803938,34.253876,0.551761,14526,0.948104,AB_1ABA060-9.nii
372,90,89.364154,3387.776044,36.945931,0.704182,18267,1.192277,AB_1ABA060-90.nii
373,91,94.957211,3357.120718,36.985409,0.691672,18042,1.177592,AB_1ABA060-91.nii


In [7]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results_new_script/cut_synthrad_abdomen_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)

Overall metrics (slice-based):


,mean ± std
MAE,118.834 ± 58.5118
MSE,4360.42 ± 1446.23
PSNR,36.402 ± 3.16541
SSIM,0.62013 ± 0.0722848


Metrics by body part (slice-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
AB,118.834 ± 58.5118,4360.42 ± 1446.23,36.402 ± 3.16541,0.62013 ± 0.0722848


Overall metrics (volume-based):


,mean ± std
MAE,119.202 ± 29.6626
MSE,4518.64 ± 844.641
PSNR,36.0676 ± 0.691409
SSIM,0.619675 ± 0.0309146


Metrics by body part (volume-based):


,MAE,MSE,PSNR,SSIM
body_part,,,,
AB,119.202 ± 29.6626,4518.64 ± 844.641,36.0676 ± 0.691409,0.619675 ± 0.0309146


## Fullbody

In [8]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results_new_script/cut_synthrad_allregions_final/test_50/test_metrics_over_all.csv")
df = df = load_metrics_csv(csv_path)


show_volume_slice_influence(df, volume=3)

Volume: AB_1ABA060 (slices=92, total_mask=1532110)


,weighted_mean
MAE,98.530119
MSE,4990.817790
PSNR,35.462053
SSIM,0.723664


,slice_id,MAE,MSE,PSNR,SSIM,mask_voxels,weight_pct,file_name
283,1,123.312709,3145.457928,37.268239,0.509401,13786,0.899805,AB_1ABA060-1.nii
284,10,103.509836,5196.411611,35.088043,0.635516,14589,0.952216,AB_1ABA060-10.nii
285,11,101.592144,5371.790597,34.943887,0.644633,14613,0.953783,AB_1ABA060-11.nii
286,12,101.431701,5255.100308,35.039268,0.622002,14605,0.953261,AB_1ABA060-12.nii
287,13,105.209039,5642.475696,34.730381,0.611235,14648,0.956067,AB_1ABA060-13.nii
...,...,...,...,...,...,...,...,...
370,89,85.828618,2833.658215,37.721603,0.903190,18380,1.199653,AB_1ABA060-89.nii
371,9,102.750447,5139.999105,35.135448,0.626176,14526,0.948104,AB_1ABA060-9.nii
372,90,87.496962,3080.354683,37.359071,0.899842,18267,1.192277,AB_1ABA060-90.nii
373,91,93.455326,3170.850626,37.233320,0.880419,18042,1.177592,AB_1ABA060-91.nii


In [ ]:
csv_path = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed/100results_new_script/cut_synthrad_allregions_final/test_50/test_metrics_over_all.csv")
show_metrics_for_csv(csv_path)